In [1]:
import pandas as pd
from typing import Dict, List, Tuple
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from OprFuncs import *
#from langchain.schema.runnable import RunnableSequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.agents import AgentExecutor, Tool, create_react_agent
#from langchain import hub
import re
#from modelEXT.PygalCodeComponents import PygalCodeComponents
#from langchain.output_parsers import PydanticOutputParser
from DatabaseManager import DatabaseManager
from langchain_experimental.agents import create_pandas_dataframe_agent

class DataAnalyzer:
    def __init__(self,dataframe,llm,user_id=None):
        self.dataframe = dataframe
        self.llm = llm
        self.data_info = data_infer(dataframe)
        self.data_description = data_describer(dataframe)
        self.data_sample = dataframe.head().to_string()
        self.data_cols = ", ".join(dataframe.columns)
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        
        if user_id:
            self.user_id = user_id
            self.user_context = self.db.get_user_context(user_id)
            if self.user_context:
                self.memory.append(HumanMessage(content=f"User Context: {self.user_context}"))
        else:
            self.user_context = None

    def analysis_data(self):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        analysis_template = '''
        You are a data analyst. You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_description}
        You are a **world-class Senior Data Analyst and Applied Statistician**, with deep expertise in business intelligence, behavioral data, financial analytics, and statistical modeling. I will provide you with a dataset in the form of a DataFrame, CSV, or Excel file.

        🎯 Your task is to perform a **comprehensive, statistically-sound, and executive-ready analysis** tailored for decision-makers, technical stakeholders, and strategic planners.

        ---

        ## 🧾 1. Executive Summary
        - Summarize the most important findings, using clear and impactful language.
        - Highlight how these findings affect the business, strategy, or operations.
        - Include headline numbers (KPIs, revenue impact, user behavior shifts...).

        ---

        ## 📊 2. Key Patterns & Strategic Insights
        - Explore key trends, distributions, and variable relationships.
        - Use metrics such as:
        - **Mean, Median, Std. Dev.**
        - **Correlation Coefficients**
        - **Distribution Skewness/Kurtosis**
        - **R² Score (if regression applies)**

        📌 Visuals may include histograms, bar charts, scatter plots, or heatmaps.

        ---

        ## 📐 3. Statistical Validation & Modeling
        - Apply formal **hypothesis tests** where applicable:
        - t-tests, ANOVA, Chi-square, or Z-tests.
        - Report **p-values** and **statistical significance**.
        - Build simple predictive or explanatory models:
        - Linear/Logistic Regression, Decision Trees...
        - Report key metrics:
        - **R²**, **RMSE**, **AUC**, or **F1-Score** (as appropriate).
        - Provide **Confidence Intervals** for estimates when relevant.

        📈 Clearly indicate statistically significant results and what they mean for the business.

        ---

        ## ⚠️ 4. Risks, Anomalies & Data Limitations
        - Identify:
        - Missing values
        - Outliers
        - Sampling bias or measurement error
        - Explain how each issue might impact model validity or business interpretations.
        - Suggest methods for mitigation (e.g., imputation, resampling, anomaly filtering).

        ---

        ## 🌱 5. Opportunities for Growth & Optimization
        - Identify actionable insights tied to business KPIs.
        - Use segmentation, clustering, or cross-tab analysis to discover growth potential.
        - Prioritize by impact, feasibility, and risk.

        ---

        ## 💡 6. Hidden or Surprising Insights
        - Detect any **non-obvious** trends, patterns, or behaviors.
        - Show how these findings might reveal blind spots or strategic advantages.

        ---

        ## 🧠 7. Strategic Recommendations
        - Provide **3–5 clear, data-backed actions** for decision-makers.
        - Align each with business objectives (cost savings, revenue growth, efficiency).
        - Include a “next steps” section (further data needed, A/B test, dashboard build...).

        ---

        ## 📊 Summary Table of Key Drivers

        | Category              | Factor            | Impact Level | Statistical Significance | Recommendation                      |
        |----------------------|-------------------|--------------|---------------------------|-------------------------------------|
        | 📈 High Impact       | [Variable Name]   | Strong       | ✅ p < 0.05                | [Recommended Action]               |
        | ⚠️ Low/Negative Impact | [Variable Name]   | Weak/Negative| ❌ Not significant         | [Mitigation Strategy or Ignore]    |

        ---

        ## 📌 Presentation Guidelines
        - Use professional, business-oriented language.
        - Include emojis 🎯 📈 ⚠️ 💡 💰 🔍 to enhance readability.
        - Be clear, direct, and data-driven.
        - If any part of the dataset is unclear or incomplete, ask clarifying questions before finalizing.

        Once the dataset is received, begin your advanced analysis.
        '''
        analysis_prompt = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "user_context"],
            template=analysis_template
        )
        
        analysis_chain = analysis_prompt | self.llm

        self.analysis = analysis_chain.invoke({
            "data_info": data_info,
            "data_sample": data_sample,
            "data_description": data_description,
            "user_context":self.user_context or "No prior context available"
        })

        formatted_analysis_prompt = analysis_template.format(data_info=data_info,data_sample=data_sample,
                                                            data_description=data_description,
                                                            user_context=self.user_context)
        self.memory.append(HumanMessage(content=formatted_analysis_prompt))
        self.memory.append(AIMessage(content=self.analysis))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_analysis_prompt,
                        response=self.analysis,
                        chat=False)
        self.generate_user_context()
        return self.analysis
    
    def questions_gen(self, num):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description

        question_prompt = '''
        You are a professional data analyst. Based on the following information about the dataset:
        1. Dataset Overview: {data_info}
        2. Dataset Sample: {data_sample}
        3. Data Summary: {data_description}
        4. Business Context: {user_context}

        Your task is to generate strategic investigative questions based on:
        - Your deep understanding of the data and its type.
        - Your interpretation of what the data means in the context of the provided business context.
        - Asking questions that may reveal insights, gaps, or opportunities that could be exploited.
        - Additionally, consider the following:
            - How could the current trends in the data impact future business decisions or strategies?
            - What potential future outcomes or projections can be made from this dataset based on existing patterns?
            - Are there any trends in the data that suggest upcoming risks or opportunities?
            - Can you identify any correlations or causal relationships that may impact future developments in the business or industry?

        Please formulate questions related to the following aspects:
        - Patterns or trends observed in the data.
        - Any relationships between columns or between the data.
        - Potential opportunities for improving business decisions or strategies based on the data.
        - Any problems or risks that might arise based on the data analysis.
        '''
        question_template = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "user_context"],
            template=question_prompt
        )

        # Create the LLM chain
        question_chain = LLMChain(llm=self.llm, prompt=question_template)

        try:
            # Run the chain to generate questions
            generated_questions = question_chain.run(data_info=data_info, 
                                                    data_sample=data_sample, 
                                                    data_description=data_description,
                                                    user_context="No prior context available")

            # Ensure the response is properly encoded and strip unnecessary spaces
            if isinstance(generated_questions, str):
                generated_questions = generated_questions.encode('utf-8', 'replace').decode('utf-8')

            print("Raw LLM Output:", repr(generated_questions))

            if not generated_questions.strip():
                print("Warning: LLM did not generate any questions.")
                return []

            # Extract questions using the helper function
            questions_list = self.extract_questions(generated_questions)

            print("Extracted Questions List:", questions_list)

            # Organize questions into a more readable format with numbers
            organized_questions = []
            for idx, question in enumerate(questions_list[:num]):
                organized_questions.append(f"{idx + 1}. {question.strip()}")

            print("Organized Questions:", organized_questions)

            # Save the generated questions to memory
            formatted_question_prompt = question_template.format(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description,
                user_context="No prior context available"
            )
            self.memory.append(HumanMessage(content=formatted_question_prompt))
            self.memory.append(AIMessage(content="\n".join(organized_questions)))
            self.db.saveMemory(reportID=self.report_id,
                            llm=self.db.llm_id_by_name(self.llm.model),
                            prompet=formatted_question_prompt,
                            response="\n".join(organized_questions),
                            chat=False)

            return organized_questions

        except Exception as e:
            print(f"Error generating questions: {str(e)}")
            return []
        
    
    def generate_recommendations(self, num_recommendations: int = 5):
        data_info = self.data_info
        data_sample = self.data_sample
        data_description = self.data_description
        analysis = self.analysis  # التحليل الذي تم عمله سابقاً

        recommendation_prompt = '''
        You are a world-class business consultant and data analyst.

        You have analyzed the following:
        - Dataset metadata: {data_info}
        - Dataset sample: {data_sample}
        - Dataset summary: {data_description}
        - Detailed business analysis: {analysis}
        - User context: {user_context}

        Based on your deep understanding of the data and analysis:
        Your task is to generate {num_recommendations} highly actionable, strategic recommendations for the business.

        Your recommendations must:
        - Be directly based on the analysis and insights.
        - Address clear business actions (e.g., optimize processes, launch new products, reduce risks, target specific segments, etc.)
        - Be specific, impactful, and feasible.
        - Cover both short-term quick wins and long-term strategic moves.
        - Include estimated expected outcome in percentage (%) where appropriate.
        - Include any potential risks or challenges for each recommendation.
        - Reference relevant metrics or insights from the analysis if possible.
        - Use professional, executive-level language.
        - Add an appropriate emoji based on risk level:
            - ✅ for Low risk
            - ⚠️ for Medium risk
            - ❗for High risk

        Output Format:

        ### 📋 Recommendations Table

        | # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
        |---|-----------------------|---------------------|-----------------------------|
        | 1 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | 2 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | ... | ... | ... | ... |

        ---

        ### 📋 Full Recommendation Details

        1. **[Recommendation Title]** [Emoji]
        - **Details:** Explain clearly what should be done and why.
        - **Expected Impact:** [e.g., Increase attendance by 10%]
        - **Metrics Reference:** [Reference specific metric if available, e.g., matches with <50% attendance]
        - **Potential Risks:** [Possible challenges or risks involved]
        - **Timeline:** [Short-term or Long-term]

        Repeat similarly for each recommendation.
        '''

        
        rec_template = PromptTemplate(
            input_variables=["data_info", "data_sample", "data_description", "analysis", "user_context", "num_recommendations"],
            template=recommendation_prompt
        )

        rec_chain = LLMChain(llm=self.llm, prompt=rec_template)

        rec_response = rec_chain.run(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            user_context=self.user_context or "No prior context available",
            num_recommendations=num_recommendations
        )

        # تسجيل في الذاكرة
        formatted_rec_prompt = recommendation_prompt.format(
            data_info=data_info,
            data_sample=data_sample,
            data_description=data_description,
            analysis=analysis,
            user_context=self.user_context or "No prior context available",
            num_recommendations=num_recommendations
        )
        self.memory.append(HumanMessage(content=formatted_rec_prompt))
        self.memory.append(AIMessage(content=rec_response))
        self.db.saveMemory(reportID=self.report_id,
                        llm=self.db.llm_id_by_name(self.llm.model),
                        prompet=formatted_rec_prompt,
                        response=rec_response,
                        chat=False)

        return rec_response


In [1]:
import pandas as pd
import logging
import json
import os
from typing import Dict, List, Tuple, Optional, Any, Union
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from pathlib import Path
from OprFuncs import data_infer, data_describer, extract_questions
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.language_models.base import BaseLanguageModel
import re
from DatabaseManager import DatabaseManager
from functools import lru_cache

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("data_analyzer.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class DataAnalyzer:
    """
    A comprehensive data analysis tool using LLMs to generate insights, 
    questions, recommendations, and visualizations from dataframes.
    """
    
    def __init__(
        self, 
        dataframe: pd.DataFrame, 
        llm: BaseLanguageModel, 
        user_id: Optional[str] = None,
        prompt_dir: str = "prompts"
    ):
        """
        Initialize the DataAnalyzer with a dataframe and language model.
        
        Args:
            dataframe: The pandas DataFrame to analyze
            llm: The language model to use for analysis
            user_id: Optional user ID for personalization
            prompt_dir: Directory containing prompt templates
        """
        self.dataframe = dataframe
        self.llm = llm
        self.data_info = data_infer(dataframe)
        self.data_description = data_describer(dataframe)
        self.data_sample = dataframe.head().to_string()
        self.data_cols = ", ".join(dataframe.columns)
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        self.user_id = user_id
        self.prompt_dir = Path(prompt_dir)
        
        # Create prompts directory if it doesn't exist
        os.makedirs(self.prompt_dir, exist_ok=True)
        
        # Extract prompt templates from code and save to files if they don't exist
        self._initialize_prompt_templates()

    def _initialize_prompt_templates(self) -> None:
        """Extract embedded prompts and save them to files for better maintainability."""
        # Define prompt templates and their file names
        prompts = {
            "analysis_template": self._get_analysis_template(),
            "question_template": self._get_question_template(),
            "recommendation_template": self._get_recommendation_template(),
            "chart_type_template": self._get_chart_type_template(),
            "columns_template": self._get_columns_template()
        }
        
        # Save prompts to files if they don't exist
        for name, content in prompts.items():
            prompt_file = self.prompt_dir / f"{name}.txt"
            if not prompt_file.exists():
                with open(prompt_file, 'w', encoding='utf-8') as f:
                    f.write(content)
                logger.info(f"Created prompt template file: {prompt_file}")

    def _get_analysis_template(self) -> str:
        """Return the analysis prompt template."""
        return '''
        You are a data analyst. You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_description}
        You are a **world-class Senior Data Analyst and Applied Statistician**, with deep expertise in business intelligence, behavioral data, financial analytics, and statistical modeling. I will provide you with a dataset in the form of a DataFrame, CSV, or Excel file.

        🎯 Your task is to perform a **comprehensive, statistically-sound, and executive-ready analysis** tailored for decision-makers, technical stakeholders, and strategic planners.

        ---

        ## 🧾 1. Executive Summary
        - Summarize the most important findings, using clear and impactful language.
        - Highlight how these findings affect the business, strategy, or operations.
        - Include headline numbers (KPIs, revenue impact, user behavior shifts...).

        ---

        ## 📊 2. Key Patterns & Strategic Insights
        - Explore key trends, distributions, and variable relationships.
        - Use metrics such as:
        - **Mean, Median, Std. Dev.**
        - **Correlation Coefficients**
        - **Distribution Skewness/Kurtosis**
        - **R² Score (if regression applies)**

        📌 Visuals may include histograms, bar charts, scatter plots, or heatmaps.

        ---

        ## 📐 3. Statistical Validation & Modeling
        - Apply formal **hypothesis tests** where applicable:
        - t-tests, ANOVA, Chi-square, or Z-tests.
        - Report **p-values** and **statistical significance**.
        - Build simple predictive or explanatory models:
        - Linear/Logistic Regression, Decision Trees...
        - Report key metrics:
        - **R²**, **RMSE**, **AUC**, or **F1-Score** (as appropriate).
        - Provide **Confidence Intervals** for estimates when relevant.

        📈 Clearly indicate statistically significant results and what they mean for the business.

        ---

        ## ⚠️ 4. Risks, Anomalies & Data Limitations
        - Identify:
        - Missing values
        - Outliers
        - Sampling bias or measurement error
        - Explain how each issue might impact model validity or business interpretations.
        - Suggest methods for mitigation (e.g., imputation, resampling, anomaly filtering).

        ---

        ## 🌱 5. Opportunities for Growth & Optimization
        - Identify actionable insights tied to business KPIs.
        - Use segmentation, clustering, or cross-tab analysis to discover growth potential.
        - Prioritize by impact, feasibility, and risk.

        ---

        ## 💡 6. Hidden or Surprising Insights
        - Detect any **non-obvious** trends, patterns, or behaviors.
        - Show how these findings might reveal blind spots or strategic advantages.

        ---

        ## 🧠 7. Strategic Recommendations
        - Provide **3–5 clear, data-backed actions** for decision-makers.
        - Align each with business objectives (cost savings, revenue growth, efficiency).
        - Include a "next steps" section (further data needed, A/B test, dashboard build...).

        ---

        ## 📊 Summary Table of Key Drivers

        | Category              | Factor            | Impact Level | Statistical Significance | Recommendation                      |
        |----------------------|-------------------|--------------|---------------------------|-------------------------------------|
        | 📈 High Impact       | [Variable Name]   | Strong       | ✅ p < 0.05                | [Recommended Action]               |
        | ⚠️ Low/Negative Impact | [Variable Name]   | Weak/Negative| ❌ Not significant         | [Mitigation Strategy or Ignore]    |

        ---

        ## 📌 Presentation Guidelines
        - Use professional, business-oriented language.
        - Include emojis 🎯 📈 ⚠️ 💡 💰 🔍 to enhance readability.
        - Be clear, direct, and data-driven.
        - If any part of the dataset is unclear or incomplete, ask clarifying questions before finalizing.

        Once the dataset is received, begin your advanced analysis.
        '''

    def _get_question_template(self) -> str:
        """Return the question generation prompt template."""
        return '''
        You are a senior data analyst hired by a company to extract meaningful, high-level, and actionable business insights from the following dataset.

        Your job is to generate advanced **strategic questions** that:
        - Are deeply rooted in the data structure and semantics.
        - Reflect important **business objectives**, patterns, risks, or growth opportunities.
        - Are **strong, insightful, and relevant** to decision-makers like company owners or managers.
        - Can be **easily visualized** using bar charts, line plots, histograms, scatter plots, or pie charts.

        **DO NOT generate general or surface-level questions. Instead, focus on questions that:**
        - Quantify change over time or between groups.
        - Explore distribution, frequency, or correlation.
        - Investigate trends, seasonality, or anomalies.
        - Provide guidance for optimizing business performance or identifying risks.

        You MUST generate exactly {num} chartable, insightful questions.

        ### INPUTS:
        1. Dataset Overview: {data_info}
        2. Dataset Sample: {data_sample}
        3. Data Summary: {data_description}

        ### OUTPUT FORMAT:
        Write {num} powerful analytical questions that:
        - Could be visualized with a chart.
        - Have clear business relevance.
        - Reflect advanced reasoning.

        Each question should be written on a separate line.

        Example Questions:
        - How has the conversion rate changed over time across different marketing channels?
        - Which regions have shown the fastest growth in revenue over the past year?
        - What is the correlation between customer satisfaction scores and return frequency?
        - How does the average transaction value vary by customer segment?
        '''

    def _get_recommendation_template(self) -> str:
        """Return the recommendation prompt template."""
        return '''
        You are a world-class business consultant and data analyst.

        You have analyzed the following:
        - Dataset metadata: {data_info}
        - Dataset sample: {data_sample}
        - Dataset summary: {data_description}
        - Detailed business analysis: {analysis}

        Based on your deep understanding of the data and analysis:
        Your task is to generate {num_recommendations} highly actionable, strategic recommendations for the business.

        Your recommendations must:
        - Be directly based on the analysis and insights.
        - Address clear business actions (e.g., optimize processes, launch new products, reduce risks, target specific segments, etc.)
        - Be specific, impactful, and feasible.
        - Cover both short-term quick wins and long-term strategic moves.
        - Include estimated expected outcome in percentage (%) where appropriate.
        - Include any potential risks or challenges for each recommendation.
        - Reference relevant metrics or insights from the analysis if possible.
        - Use professional, executive-level language.
        - Add an appropriate emoji based on risk level:
            - ✅ for Low risk
            - ⚠️ for Medium risk
            - ❗for High risk

        Output Format:

        ### 📋 Recommendations Table

        | # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
        |---|-----------------------|---------------------|-----------------------------|
        | 1 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | 2 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
        | ... | ... | ... | ... |

        ---

        ### 📋 Full Recommendation Details

        1. **[Recommendation Title]** [Emoji]
        - **Details:** Explain clearly what should be done and why.
        - **Expected Impact:** [e.g., Increase attendance by 10%]
        - **Metrics Reference:** [Reference specific metric if available, e.g., matches with <50% attendance]
        - **Potential Risks:** [Possible challenges or risks involved]
        - **Timeline:** [Short-term or Long-term]

        Repeat similarly for each recommendation.
        '''

    def _get_chart_type_template(self) -> str:
        """Return the chart type selection prompt template."""
        return '''You are an expert at selecting chart types for data visualization. Strictly follow these rules:
            
            1. CHART SELECTION GUIDE:
            - For comparing categories: Bar 
            - For trends over time: Line
            - For parts of a whole: Pie (few categories)
            - For relationships: Scatter
            - For the distribution of a numerical variable: Histogram
            
            3. OUTPUT FORMAT (EXACTLY):
            chart_type: [Bar|Line|Pie|Scatter|Histogram]
            
            Data Description: {data_description}
            Available Columns: {columns}
            Sample Data: {sample_data}
            Question: {question}
            
            Respond ONLY with:
            chart_type: [chart_type]'''

    def _get_columns_template(self) -> str:
        """Return the column selection prompt template."""
        return '''You are an expert at selecting relevant columns for data visualization. Strictly follow:
            
            1. COLUMN SELECTION RULES:
            - Focus on columns mentioned in the question
            - What is being measured (numerical columns)
            - What is being compared/grouped by (categorical columns)
            - Any time dimensions for trends
            - Never suggest columns not in Available Columns
            
            2. OUTPUT FORMAT (EXACTLY):
            columns: [exact_column_name1, exact_column_name2]
            
            Data Description: {data_description}
            Available Columns: {columns}
            Sample Data: {sample_data}
            Question: {question}
            
            Respond ONLY with:
            columns: [column1, column2]'''

    def _load_prompt_template(self, template_name: str) -> str:
        """
        Load a prompt template from file or use the default embedded one.
        
        Args:
            template_name: Name of the template to load
            
        Returns:
            The prompt template as a string
        """
        template_file = self.prompt_dir / f"{template_name}.txt"
        try:
            if template_file.exists():
                with open(template_file, 'r', encoding='utf-8') as f:
                    return f.read()
            else:
                # Fallback to embedded template method
                method_name = f"_get_{template_name}"
                if hasattr(self, method_name):
                    return getattr(self, method_name)()
                else:
                    logger.warning(f"Template {template_name} not found")
                    return ""
        except Exception as e:
            logger.error(f"Error loading prompt template {template_name}: {str(e)}")
            # Fallback to embedded template
            method_name = f"_get_{template_name}"
            if hasattr(self, method_name):
                return getattr(self, method_name)()
            return ""

    def _save_to_memory(self, prompt: str, response: str, is_chat: bool = False) -> None:
        """
        Save interactions to memory and database.
        
        Args:
            prompt: The prompt sent to the LLM
            response: The response from the LLM
            is_chat: Whether this is a chat interaction
        """
        try:
            self.memory.append(HumanMessage(content=prompt))
            self.memory.append(AIMessage(content=response))
            
            if self.report_id is not None:
                self.db.saveMemory(
                    reportID=self.report_id,
                    llm=self.db.llm_id_by_name(self.llm.model),
                    prompet=prompt,  # Note: typo in original API
                    response=response,
                    chat=is_chat
                )
        except Exception as e:
            logger.error(f"Error saving to memory: {str(e)}")

    def analysis_data(self) -> str:
        """
        Perform comprehensive data analysis using the language model.
        
        Returns:
            Detailed analysis text
        """
        try:
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description

            # Load the analysis template
            analysis_template = self._load_prompt_template("analysis_template")
            
            # Create the prompt
            analysis_prompt = PromptTemplate(
                input_variables=["data_info", "data_sample", "data_description"],
                template=analysis_template
            )
            
            # Build the chain with the modern pattern
            analysis_chain = analysis_prompt | self.llm

            # Run the analysis
            logger.info("Starting data analysis...")
            self.analysis = analysis_chain.invoke({
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description
            })
            logger.info("Data analysis completed")

            # Format the prompt for memory
            formatted_analysis_prompt = analysis_template.format(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            
            # Save to memory
            self._save_to_memory(formatted_analysis_prompt, self.analysis)
            
            return self.analysis
        
        except Exception as e:
            logger.error(f"Error in analysis_data: {str(e)}")
            return f"Error performing data analysis: {str(e)}"

    def questions_gen(self, num: int) -> List[str]:
        """
        Generate insightful questions about the data.
        
        Args:
            num: Number of questions to generate
            
        Returns:
            List of generated questions
        """
        try:
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description

            # Load question template
            question_template_str = self._load_prompt_template("question_template")
            
            # Create prompt template
            question_template = PromptTemplate(
                input_variables=["num", "data_info", "data_sample", "data_description"],
                template=question_template_str
            )

            # Create chain
            question_chain = question_template | self.llm

            # Generate questions
            logger.info(f"Generating {num} questions...")
            generated_questions = question_chain.invoke({
                "num": num,
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description
            })

            # Ensure proper encoding
            if isinstance(generated_questions, str):
                generated_questions = generated_questions.encode('utf-8', 'replace').decode('utf-8')

            logger.debug(f"Raw LLM Output: {repr(generated_questions)}")

            if not generated_questions.strip():
                logger.warning("LLM did not generate any questions.")
                return []

            # Extract questions
            questions_list = extract_questions(generated_questions)
            logger.info(f"Extracted {len(questions_list)} questions")

            # Trim or handle missing questions
            if len(questions_list) > num:
                questions_list = questions_list[:num]
                logger.info(f"Trimmed to {num} questions")
            elif len(questions_list) < num:
                logger.warning(f"Expected {num} questions, but got {len(questions_list)}")

            # Format prompt for memory
            formatted_question_prompt = question_template.format(
                num=num,
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            
            # Save to memory
            self._save_to_memory(
                formatted_question_prompt, 
                "\n".join(questions_list)
            )

            return questions_list

        except Exception as e:
            logger.error(f"Error generating questions: {str(e)}")
            return []

    def chat(self, question: str) -> str:
        """
        Chat with the data analysis system.
        
        Args:
            question: The user's question
            
        Returns:
            The model's response
        """
        try:
            # Create chat prompt
            prompt_template = ChatPromptTemplate.from_messages([
                ("system", "You are a data analyst."),
                MessagesPlaceholder(variable_name="memory"),
                ("human", "{input}")
            ])
            
            # Create chain
            chain = prompt_template | self.llm

            # Get response
            logger.info(f"Processing chat: {question[:50]}...")
            response = chain.invoke({
                "input": question, 
                "memory": self.memory
            })
            
            # Save to memory
            self._save_to_memory(question, response, is_chat=True)
            
            return response
            
        except Exception as e:
            logger.error(f"Error in chat: {str(e)}")
            return f"Sorry, I encountered an error: {str(e)}"

    @lru_cache(maxsize=32)
    def select_chart_type(self, question: str) -> str:
        """
        Select an appropriate chart type for visualizing data based on the question.
        
        Args:
            question: The analytical question
            
        Returns:
            Recommended chart type
        """
        try:
            # Load chart type template
            chart_type_template = self._load_prompt_template("chart_type_template")
            
            # Create prompt
            self.chart_type_prompt = ChatPromptTemplate.from_messages([
                ("system", chart_type_template)
            ])

            # Temporarily reduce temperature for more consistent results
            original_temp = getattr(self.llm, 'temperature', None)
            if hasattr(self.llm, 'temperature'):
                self.llm.temperature = 0.3
            
            # Create chain
            chain = self.chart_type_prompt | self.llm
            
            # Get response
            response = chain.invoke({
                "data_description": self.data_description,
                "columns": self.data_cols,
                "sample_data": self.data_sample,
                "question": question
            })
            
            # Restore temperature
            if hasattr(self.llm, 'temperature') and original_temp is not None:
                self.llm.temperature = original_temp
            
            # Parse response
            chart_match = re.search(r'chart_type:\s*([a-zA-Z]+)', response, re.IGNORECASE)
            chart_type = chart_match.group(1) if chart_match else None
            
            # Validate against allowed chart types
            allowed_charts = {
                'Bar', 'Line', 'Histogram', 
                'Pie', 'Scatter'
            }
            
            return chart_type if chart_type in allowed_charts else 'Bar'
            
        except Exception as e:
            logger.error(f"Error selecting chart type: {str(e)}")
            return 'Bar'  # Default to bar chart on error

    @lru_cache(maxsize=32)
    def select_columns(self, question: str) -> List[str]:
        """
        Select relevant columns for a visualization based on the question.
        
        Args:
            question: The analytical question
            
        Returns:
            List of column names
        """
        try:
            # Load columns template
            columns_template = self._load_prompt_template("columns_template")
            
            # Create prompt
            self.columns_prompt = ChatPromptTemplate.from_messages([
                ("system", columns_template)
            ])

            # Temporarily reduce temperature
            original_temp = getattr(self.llm, 'temperature', None)
            if hasattr(self.llm, 'temperature'):
                self.llm.temperature = 0.3
            
            # Create chain
            chain = self.columns_prompt | self.llm
            
            # Get response
            response = chain.invoke({
                "data_description": self.data_description,
                "columns": self.data_cols,
                "sample_data": self.data_sample,
                "question": question
            })
            
            # Restore temperature
            if hasattr(self.llm, 'temperature') and original_temp is not None:
                self.llm.temperature = original_temp
            
            # Parse response
            cols_match = re.search(r'columns:\s*\[([^\]]+)\]', response)
            if cols_match:
                columns = [col.strip() for col in cols_match.group(1).split(',')]
            else:
                # Fallback parsing
                cols_line = next((line for line in response.split('\n') if line.startswith('columns:')), '')
                columns = [col.strip() for col in cols_line.replace('columns:', '').split(',') if col.strip()]
            
            # Validate columns exist in data
            available_cols = self.dataframe.columns.tolist()
            valid_columns = [col for col in columns if col in available_cols]
            
            if not valid_columns and columns:
                logger.warning(f"None of the suggested columns {columns} exist in the dataframe")
            
            return valid_columns
            
        except Exception as e:
            logger.error(f"Error selecting columns: {str(e)}")
            return []

    def get_chart_recommendation(self, question: str) -> Tuple[str, List[str]]:
        """
        Get a combined chart type and columns recommendation.
        
        Args:
            question: The analytical question
            
        Returns:
            Tuple of (chart_type, columns_list)
        """
        chart_type = self.select_chart_type(question)
        columns = self.select_columns(question)
        return chart_type, columns

    def generate_recommendations(self, num_recommendations: int = 5) -> str:
        """
        Generate business recommendations based on the data analysis.
        
        Args:
            num_recommendations: Number of recommendations to generate
            
        Returns:
            Formatted recommendations text
        """
        try:
            # Check if we have analysis data
            if not hasattr(self, 'analysis') or not self.analysis:
                logger.warning("No analysis available. Running analysis first.")
                self.analysis_data()
            
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description
            analysis = self.analysis
            
            # Load recommendation template
            recommendation_template = self._load_prompt_template("recommendation_template")
            
            # Create prompt template
            rec_template = PromptTemplate(
                input_variables=["data_info", "data_sample", "data_description", "analysis", "num_recommendations"],
                template=recommendation_template
            )

            # Use modern pattern instead of deprecated LLMChain
            rec_chain = rec_template | self.llm
            
            # Generate recommendations
            logger.info(f"Generating {num_recommendations} recommendations...")
            rec_response = rec_chain.invoke({
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description,
                "analysis": analysis,
                "num_recommendations": num_recommendations
            })
            
            # Format prompt for memory
            formatted_rec_prompt = recommendation_template.format(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description,
                analysis=analysis,
                num_recommendations=num_recommendations
            )
            
            # Save to memory
            self._save_to_memory(formatted_rec_prompt, rec_response)
            
            return rec_response
            
        except Exception as e:
            logger.error(f"Error generating recommendations: {str(e)}")
            return f"Error generating recommendations: {str(e)}"
    
    def export_analysis(self, format_type: str = 'json', output_path: Optional[str] = None) -> Union[str, Dict[str, Any]]:
        """
        Export the analysis results to a specified format.
        
        Args:
            format_type: The format to export to ('json', 'md', 'txt')
            output_path: Path to save the exported file
            
        Returns:
            Either the path to the saved file or the data structure
        """
        try:
            # Check if we have analysis data
            if not hasattr(self, 'analysis') or not self.analysis:
                logger.warning("No analysis available. Running analysis first.")
                self.analysis_data()
            
            # Prepare the export data
            export_data = {
                'data_info': self.data_info,
                'analysis': self.analysis,
            }
            
            # Add recommendations if available
            if hasattr(self, 'recommendations'):
                export_data['recommendations'] = self.recommendations
            
            # Process based on format type
            if format_type.lower() == 'json':
                result = json.dumps(export_data, indent=2)
                
                if output_path:
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.write(result)
                    return output_path
                return export_data
                
            elif format_type.lower() == 'md':
                md_content = f"# Data Analysis Report\n\n## Data Information\n\n```\n{self.data_info}\n```\n\n"
                md_content += f"## Analysis\n\n{self.analysis}\n\n"
                
                if hasattr(self, 'recommendations'):
                    md_content += f"## Recommendations\n\n{self.recommendations}\n"
                
                if output_path:
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.write(md_content)
                    return output_path
                return md_content
                
            elif format_type.lower() == 'txt':
                txt_content = f"DATA ANALYSIS REPORT\n\nDATA INFORMATION\n\n{self.data_info}\n\n"
                txt_content += f"ANALYSIS\n\n{self.analysis}\n\n"
                
                if hasattr(self, 'recommendations'):
                    txt_content += f"RECOMMENDATIONS\n\n{self.recommendations}\n"
                
                if output_path:
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.write(txt_content)
                    return output_path
                return txt_content
                
            else:
                raise ValueError(f"Unsupported export format: {format_type}")
                
        except Exception as e:
            logger.error(f"Error exporting analysis: {str(e)}")
            return f"Error exporting analysis: {str(e)}"

In [2]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.analysis_data()

# Print the analysis
print(analysis_result)

2025-05-13 22:41:58,364 - httpx - INFO - HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


As a world-class Senior Data Analyst and Applied Statistician, I will perform a comprehensive, statistically-sound, and executive-ready analysis tailored for decision-makers, technical stakeholders, and strategic planners. Here's my analysis based on the provided dataset:

**🧾 1. Executive Summary**

* The most important finding is that there is no significant correlation between Home Team Goals and Attendance (R² = 0.02). This suggests that attendance may not be a reliable indicator of home team performance.
* Another key insight is that the mean number of Home Team Goals per match is relatively low at 1.81, indicating that matches are often closely contested.
* The data also shows that the majority of matches (75%) have less than 5000 attendees, which may indicate that smaller stadiums or regional venues are common.

**Headline numbers:**

* Mean Attendance: 45,164
* Median Home Team Goals: 2

**📊 2. Key Patterns & Strategic Insights**

* The distribution of Attendance is skewed to t

In [ ]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
analyzer = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
analysis_result = analyzer.questions_gen(num=5)

# Print the analysis
print(analysis_result)


Raw LLM Output: 'Here are five analysis questions about the dataset:\n\n1. What is the average attendance by stage in the tournament?\n2. Which team scored the most goals during half-time, and what was their overall record?\n3. Is there a correlation between home team goals and win conditions?\n4. How does the distribution of away team goals vary across different cities (Montevideo)?\n5. What is the relationship between RoundID and MatchID?'
Extracted Questions List: ['What is the average attendance by stage in the tournament?', 'Which team scored the most goals during half-time, and what was their overall record?', 'Is there a correlation between home team goals and win conditions?', 'How does the distribution of away team goals vary across different cities (Montevideo)?', 'What is the relationship between RoundID and MatchID?']
['What is the average attendance by stage in the tournament?', 'Which team scored the most goals during half-time, and what was their overall record?', 'Is th

In [17]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
question = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis
question_result = question.questions_gen(num=5)

# Print the analysis
print(question_result)

Raw LLM Output: 'Here are five analysis questions about the dataset:\n\n1. What is the average attendance by year?\n2. Are there any significant differences in home team goals scored between different stages (Group 1, Group 2, etc.)?\n3. How does the distribution of half-time home goals compare to that of away goals?\n4. Is there a correlation between win conditions and match outcome (win/loss)?\n5. What is the average attendance for matches with high-scoring games (i.e., games with more than 3 total goals)?'
Extracted Questions List: ['What is the average attendance by year?', 'Are there any significant differences in home team goals scored between different stages (Group 1, Group 2, etc.)?', 'How does the distribution of half-time home goals compare to that of away goals?', 'Is there a correlation between win conditions and match outcome (win/loss)?', 'What is the average attendance for matches with high-scoring games (i.e., games with more than 3 total goals)?']
['What is the averag

In [2]:
import pandas as pd
from DataAnalyzer import DataAnalyzer 
from langchain_ollama import OllamaLLM

# Load the data
data = pd.read_csv("WorldCupMatches/WorldCupMatches.csv")
df = pd.DataFrame(data)

# Load the model
llm = OllamaLLM(model='llama3')

# Create analyzer object
recommandation = DataAnalyzer(dataframe=df, llm=llm, user_id='huss')

# Run analysis first
recommandation.analysis_data()

# Then generate recommendations
recommandation_result = recommandation.generate_recommendations(5)

# Print the recommendations
print(recommandation_result)


d:\My-Githup\Axiora\DataAnalyzer.py:443: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  rec_chain = LLMChain(llm=self.llm, prompt=rec_template)
d:\My-Githup\Axiora\DataAnalyzer.py:445: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rec_response = rec_chain.run(


### 📋 Recommendations Table

| # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
|---|-----------------------|---------------------|-----------------------------|
| 1 | Optimize Home Team Performance | 15% increase in home team wins | ✅ Low risk of overestimating opponents |
| 2 | Enhance Away Team Strategy | 8% increase in away team points scored | ⚠️ Medium risk of losing key players |
| 3 | Launch Fan Engagement Campaign | 12% increase in attendance for high-profile matches | ❗ High risk of alienating existing fans |
| 4 | Implement Data-Driven Coaching | 10% increase in win rate for top-performing teams | ✅ Low risk of misinterpreting data insights |
| 5 | Invest in Player Development Program | 11% increase in player value over the next two seasons | ⚠️ Medium risk of not attracting top talent |

### 📋 Full Recommendation Details

1. **Optimize Home Team Performance** ✅
- **Details:** Develop a customized coaching strategy for each home team, focusing o

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional, Any, Union
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from OprFuncs import *
#from langchain.schema.runnable import RunnableSequence
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
#from langchain.agents import AgentExecutor, Tool, create_react_agent
#from langchain import hub
import re
#from modelEXT.PygalCodeComponents import PygalCodeComponents
#from langchain.output_parsers import PydanticOutputParser
from DatabaseManager import DatabaseManager
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import logging
import json
import os
import pickle
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class DataAnalyzer:
    """
    A comprehensive data analysis system that uses LLMs to generate insights,
    visualizations, and recommendations from datasets.
    """
    
    # Available chart types
    CHART_TYPES = {'Bar', 'Line', 'Histogram', 'Pie', 'Scatter', 'Heatmap', 'Box', 'Area'}
    
    def __init__(self, dataframe, llm, user_id=None):
        """
        Initialize the DataAnalyzer with a dataframe and language model.
        
        Args:
            dataframe: Pandas DataFrame containing the data to analyze
            llm: Language model to use for analysis
            user_id: Optional user identifier for database operations
        """
        self.dataframe = dataframe
        self.original_dataframe = dataframe.copy()  # Store original data
        self.llm = llm
        try:
            self.data_info = data_infer(dataframe)
            self.data_description = data_describer(dataframe)
            self.data_sample = dataframe.head().to_string()
            self.data_cols = ", ".join(dataframe.columns)
        except Exception as e:
            logger.error(f"Error initializing data properties: {str(e)}")
            self.data_info = "Error generating data info"
            self.data_description = "Error generating data description"
            self.data_sample = "Error generating data sample"
            self.data_cols = "Error retrieving columns"
            
        self.db = DatabaseManager()
        self.report_id = None
        self.memory = []
        self.user_id = user_id
        self._original_temperature = getattr(self.llm, 'temperature', 0.7)
        self.analysis_results = {}
        self.cleaning_log = []

    def _set_temp(self, temp: float) -> None:
        """Safely set LLM temperature with original value tracking."""
        try:
            self._original_temperature = getattr(self.llm, 'temperature', self._original_temperature)
            self.llm.temperature = temp
        except AttributeError:
            logger.warning("LLM does not support temperature adjustment")
            
    def _reset_temp(self) -> None:
        """Reset LLM temperature to original value."""
        try:
            self.llm.temperature = self._original_temperature
        except AttributeError:
            logger.warning("LLM does not support temperature adjustment")

    def _save_to_memory(self, prompt: str, response: str, is_chat: bool = False) -> None:
        """Save interactions to memory and database."""
        try:
            self.memory.append(HumanMessage(content=prompt))
            self.memory.append(AIMessage(content=response))
            
            if self.report_id:
                llm_id = self.db.llm_id_by_name(self.llm.model) if hasattr(self.llm, 'model') else None
                self.db.saveMemory(
                    reportID=self.report_id,
                    llm=llm_id,
                    prompet=prompt,  # Note: typo in DB method
                    response=response,
                    chat=is_chat
                )
        except Exception as e:
            logger.error(f"Error saving to memory: {str(e)}")

    def analysis_data(self) -> str:
        """
        Perform comprehensive analysis of the dataset.
        
        Returns:
            A detailed analysis report
        """
        try:
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description

            analysis_template = '''
            You are a data analyst. You are provided with:
            1. Dataset metadata: {data_info}
            2. Dataset sample: {data_sample}
            3. Dataset summary: {data_description}
            You are a **world-class Senior Data Analyst and Applied Statistician**, with deep expertise in business intelligence, behavioral data, financial analytics, and statistical modeling. I will provide you with a dataset in the form of a DataFrame, CSV, or Excel file.

            🎯 Your task is to perform a **comprehensive, statistically-sound, and executive-ready analysis** tailored for decision-makers, technical stakeholders, and strategic planners.

            ---

            ## 🧾 1. Executive Summary
            - Summarize the most important findings, using clear and impactful language.
            - Highlight how these findings affect the business, strategy, or operations.
            - Include headline numbers (KPIs, revenue impact, user behavior shifts...).

            ---

            ## 📊 2. Key Patterns & Strategic Insights
            - Explore key trends, distributions, and variable relationships.
            - Use metrics such as:
            - **Mean, Median, Std. Dev.**
            - **Correlation Coefficients**
            - **Distribution Skewness/Kurtosis**
            - **R² Score (if regression applies)**

            📌 Visuals may include histograms, bar charts, scatter plots, or heatmaps.

            ---

            ## 📐 3. Statistical Validation & Modeling
            - Apply formal **hypothesis tests** where applicable:
            - t-tests, ANOVA, Chi-square, or Z-tests.
            - Report **p-values** and **statistical significance**.
            - Build simple predictive or explanatory models:
            - Linear/Logistic Regression, Decision Trees...
            - Report key metrics:
            - **R²**, **RMSE**, **AUC**, or **F1-Score** (as appropriate).
            - Provide **Confidence Intervals** for estimates when relevant.

            📈 Clearly indicate statistically significant results and what they mean for the business.

            ---

            ## ⚠️ 4. Risks, Anomalies & Data Limitations
            - Identify:
            - Missing values
            - Outliers
            - Sampling bias or measurement error
            - Explain how each issue might impact model validity or business interpretations.
            - Suggest methods for mitigation (e.g., imputation, resampling, anomaly filtering).

            ---

            ## 🌱 5. Opportunities for Growth & Optimization
            - Identify actionable insights tied to business KPIs.
            - Use segmentation, clustering, or cross-tab analysis to discover growth potential.
            - Prioritize by impact, feasibility, and risk.

            ---

            ## 💡 6. Hidden or Surprising Insights
            - Detect any **non-obvious** trends, patterns, or behaviors.
            - Show how these findings might reveal blind spots or strategic advantages.

            ---

            ## 🧠 7. Strategic Recommendations
            - Provide **3–5 clear, data-backed actions** for decision-makers.
            - Align each with business objectives (cost savings, revenue growth, efficiency).
            - Include a "next steps" section (further data needed, A/B test, dashboard build...).

            ---

            ## 📊 Summary Table of Key Drivers

            | Category              | Factor            | Impact Level | Statistical Significance | Recommendation                      |
            |----------------------|-------------------|--------------|---------------------------|-------------------------------------|
            | 📈 High Impact       | [Variable Name]   | Strong       | ✅ p < 0.05                | [Recommended Action]               |
            | ⚠️ Low/Negative Impact | [Variable Name]   | Weak/Negative| ❌ Not significant         | [Mitigation Strategy or Ignore]    |

            ---

            ## 📌 Presentation Guidelines
            - Use professional, business-oriented language.
            - Include emojis 🎯 📈 ⚠️ 💡 💰 🔍 to enhance readability.
            - Be clear, direct, and data-driven.
            - If any part of the dataset is unclear or incomplete, ask clarifying questions before finalizing.

            Once the dataset is received, begin your advanced analysis.
            '''

            analysis_prompt = PromptTemplate(
                input_variables=["data_info", "data_sample", "data_description"],
                template=analysis_template
            )
            
            analysis_chain = analysis_prompt | self.llm

            self.analysis = analysis_chain.invoke({
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description
            })

            formatted_analysis_prompt = analysis_template.format(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            
            self._save_to_memory(formatted_analysis_prompt, self.analysis)
            return self.analysis
            
        except Exception as e:
            error_msg = f"Error performing data analysis: {str(e)}"
            logger.error(error_msg)
            return error_msg

    def extract_questions(self, text: str) -> List[str]:
        """
        Extract questions from the LLM's response.
        
        Args:
            text: Text containing questions
            
        Returns:
            List of extracted questions
        """
        try:
            # Try to extract questions using multiple patterns
            # Pattern 1: Look for numbered questions (1. What is...)
            numbered_pattern = r'(?:\d+\.|\-)\s*(.*?)\s*(?=\d+\.|\-|$)'
            numbered_matches = re.findall(numbered_pattern, text, re.DOTALL)
            
            # Pattern 2: Look for questions with question marks
            question_mark_pattern = r'([^.!?]+\?)'
            question_mark_matches = re.findall(question_mark_pattern, text)
            
            # Pattern 3: Line by line
            line_matches = [line.strip() for line in text.split('\n') if line.strip() and not line.startswith('#')]
            
            # Combine and clean results
            all_matches = numbered_matches or question_mark_matches or line_matches
            
            # Clean up the questions
            questions = [q.strip() for q in all_matches if q.strip()]
            
            # If we didn't find any questions using the patterns, split by newlines as a fallback
            if not questions:
                questions = [line.strip() for line in text.split('\n') if line.strip()]
                
            return questions
        except Exception as e:
            logger.error(f"Error extracting questions: {str(e)}")
            return []

    def questions_gen(self, num: int) -> List[str]:
        """
        Generate insightful questions about the dataset.
        
        Args:
            num: Number of questions to generate
            
        Returns:
            List of data analysis questions
        """
        try:
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description
            
            question_prompt = f"""
            You are a senior data analyst hired by a company to extract meaningful, high-level, and actionable business insights from the following dataset.

            Your job is to generate advanced **strategic questions** that:
            - Are deeply rooted in the data structure and semantics.
            - Reflect important **business objectives**, patterns, risks, or growth opportunities.
            - Are **strong, insightful, and relevant** to decision-makers like company owners or managers.
            - Can be **easily visualized** using bar charts, line plots, histograms, scatter plots, or pie charts.

            **DO NOT generate general or surface-level questions. Instead, focus on questions that:**
            - Quantify change over time or between groups.
            - Explore distribution, frequency, or correlation.
            - Investigate trends, seasonality, or anomalies.
            - Provide guidance for optimizing business performance or identifying risks.

            You MUST generate exactly {num} chartable, insightful questions.

            ### INPUTS:
            1. Dataset Overview: {data_info}
            2. Dataset Sample: {data_sample}
            3. Data Summary: {data_description}

            ### OUTPUT FORMAT:
            Write {num} powerful analytical questions that:
            - Could be visualized with a chart.
            - Have clear business relevance.
            - Reflect advanced reasoning.

            Each question should be written on a separate line.

            Example Questions:
            - How has the conversion rate changed over time across different marketing channels?
            - Which regions have shown the fastest growth in revenue over the past year?
            - What is the correlation between customer satisfaction scores and return frequency?
            - How does the average transaction value vary by customer segment?
            """

            question_template = PromptTemplate(
                input_variables=["num", "data_info", "data_sample", "data_description"],
                template=question_prompt
            )

            question_chain = question_template | self.llm

            generated_questions = question_chain.invoke({
                "num": num,
                "data_info": data_info,
                "data_sample": data_sample,
                "data_description": data_description
            })

            # Ensure the response is properly encoded
            if isinstance(generated_questions, str):
                generated_questions = generated_questions.encode('utf-8', 'replace').decode('utf-8')

            logger.debug(f"Raw LLM Output: {repr(generated_questions)}")

            if not generated_questions.strip():
                logger.warning("LLM did not generate any questions.")
                return []

            # Use the extraction function
            questions_list = self.extract_questions(generated_questions)

            logger.debug(f"Extracted Questions List: {questions_list}")

            # Trim or handle missing questions
            if len(questions_list) > num:
                questions_list = questions_list[:num]
            elif len(questions_list) < num:
                logger.warning(f"Expected {num} questions, but got {len(questions_list)}")

            # Store in memory
            formatted_question_prompt = question_template.format(
                num=num,
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description
            )
            
            self._save_to_memory(formatted_question_prompt, "\n".join(questions_list))
            return questions_list

        except Exception as e:
            logger.error(f"Error generating questions: {str(e)}")
            return []

    def chat(self, question: str) -> str:
        """
        Interact with the data analysis system to answer questions about the dataset.
        
        Args:
            question: User's question about the data
            
        Returns:
            The model's response with data-informed insights
        """
        try:
            system_prompt = f"""
            You are a data analyst with expertise in analyzing {self.dataframe.shape[1]} variables across {self.dataframe.shape[0]} records.

            Dataset context:
            - Type of data: {self.data_info.splitlines()[0] if self.data_info else 'Unknown dataset'}
            - Key columns: {', '.join(self.dataframe.columns[:5]) if len(self.dataframe.columns) > 5 else self.data_cols}

            Instructions:
            - Answer using ONLY the data available.
            - If asked about unknown variables, respond transparently.
            - Prioritize clarity, relevance, and helpfulness.
            """

            prompt = ChatPromptTemplate.from_messages([
                ("system", system_prompt),
                MessagesPlaceholder(variable_name="chat_history"),
                ("human", "{question}")
            ])

            chain = prompt | self.llm

            response = chain.invoke({
                "chat_history": self.memory,
                "question": question
            })

            self._save_to_memory(question, response, is_chat=True)
            return response
            
        except Exception as e:
            error_msg = f"Error in chat: {str(e)}"
            logger.error(error_msg)
            return error_msg
    
    def select_chart_type(self, question: str) -> str:
        """
        Select the most appropriate chart type based on the question.
        
        Args:
            question: The analytical question
            
        Returns:
            Recommended chart type
        """
        try:
            self.chart_type_prompt = ChatPromptTemplate.from_messages([
                ("system", f"""You are an expert at selecting chart types for data visualization. Strictly follow these rules:
                
                1. CHART SELECTION GUIDE:
                - For comparing categories: Bar 
                - For trends over time: Line
                - For parts of a whole: Pie (few categories)
                - For relationships: Scatter
                - For the distribution of a numerical variable: Histogram
                - For showing correlation matrices: Heatmap
                - For distributions and outliers: Box
                - For cumulative values over time: Area
                
                3. OUTPUT FORMAT (EXACTLY):
                chart_type: [Bar|Line|Pie|Scatter|Histogram|Heatmap|Box|Area]
                
                Data Description: {{data_description}}
                Available Columns: {{columns}}
                Sample Data: {{sample_data}}
                Question: {{question}}
                
                Respond ONLY with:
                chart_type: [chart_type]""")
            ])

            self._set_temp(0.3)
            chain = self.chart_type_prompt | self.llm
            response = chain.invoke({
                "data_description": self.data_description,
                "columns": self.data_cols,
                "sample_data": self.data_sample,
                "question": question
            })
            self._reset_temp()
            
            # Parse response
            chart_match = re.search(r'chart_type:\s*([a-zA-Z]+)', response, re.IGNORECASE)
            chart_type = chart_match.group(1) if chart_match else None
            
            # Validate
            if chart_type not in self.CHART_TYPES:
                logger.warning(f"Invalid chart type '{chart_type}', defaulting to 'Bar'")
                return 'Bar'
                
            return chart_type
            
        except Exception as e:
            logger.error(f"Error selecting chart type: {str(e)}")
            return 'Bar'  # Default to bar chart on error
    
    def select_columns(self, question: str) -> List[str]:
        """
        Select the most relevant columns for visualization based on the question.
        
        Args:
            question: The analytical question
            
        Returns:
            List of relevant column names
        """
        try:
            self.columns_prompt = ChatPromptTemplate.from_messages([
                ("system", """You are an expert at selecting relevant columns for data visualization. Strictly follow:
                
                1. COLUMN SELECTION RULES:
                - Focus on columns mentioned in the question
                - What is being measured (numerical columns)
                - What is being compared/grouped by (categorical columns)
                - Any time dimensions for trends
                - Never suggest columns not in Available Columns
                
                2. OUTPUT FORMAT (EXACTLY):
                columns: [exact_column_name1, exact_column_name2]
                
                Data Description: {data_description}
                Available Columns: {columns}
                Sample Data: {sample_data}
                Question: {question}
                
                Respond ONLY with:
                columns: [column1, column2]""")
            ])

            self._set_temp(0.3)
            chain = self.columns_prompt | self.llm
            response = chain.invoke({
                "data_description": self.data_description,
                "columns": self.data_cols,
                "sample_data": self.data_sample,
                "question": question
            })
            self._reset_temp()
            
            # Parse response
            cols_match = re.search(r'columns:\s*\[([^\]]+)\]', response)
            if cols_match:
                columns = [col.strip() for col in cols_match.group(1).split(',')]
            else:
                # Fallback parsing
                cols_line = next((line for line in response.split('\n') if line.startswith('columns:')), '')
                columns = [col.strip() for col in cols_line.replace('columns:', '').split(',') if col.strip()]
            
            # Validate columns exist in data
            available_cols = self.dataframe.columns.tolist()
            valid_columns = [col for col in columns if col in available_cols]
            
            if not valid_columns and available_cols:
                logger.warning("No valid columns selected, using first available column")
                return [available_cols[0]]
                
            return valid_columns
            
        except Exception as e:
            logger.error(f"Error selecting columns: {str(e)}")
            # Return first column as fallback
            return [self.dataframe.columns[0]] if len(self.dataframe.columns) > 0 else []
    
    def get_chart_recommendation(self, question: str) -> Tuple[str, List[str]]:
        """
        Get a complete chart recommendation including type and columns.
        
        Args:
            question: The analytical question
            
        Returns:
            Tuple of (chart_type, columns)
        """
        chart_type = self.select_chart_type(question)
        columns = self.select_columns(question)
        return chart_type, columns
    
    def generate_recommendations(self, num_recommendations: int = 5) -> str:
        """
        Generate strategic business recommendations based on data analysis.
        
        Args:
            num_recommendations: Number of recommendations to generate
            
        Returns:
            Formatted recommendations report
        """
        try:
            if not hasattr(self, 'analysis'):
                logger.warning("No analysis available. Running analysis first.")
                self.analysis_data()
                
            data_info = self.data_info
            data_sample = self.data_sample
            data_description = self.data_description
            analysis = self.analysis

            recommendation_prompt = '''
            You are a world-class business consultant and data analyst.

            You have analyzed the following:
            - Dataset metadata: {data_info}
            - Dataset sample: {data_sample}
            - Dataset summary: {data_description}
            - Detailed business analysis: {analysis}

            Based on your deep understanding of the data and analysis:
            Your task is to generate {num_recommendations} highly actionable, strategic recommendations for the business.

            Your recommendations must:
            - Be directly based on the analysis and insights.
            - Address clear business actions (e.g., optimize processes, launch new products, reduce risks, target specific segments, etc.)
            - Be specific, impactful, and feasible.
            - Cover both short-term quick wins and long-term strategic moves.
            - Include estimated expected outcome in percentage (%) where appropriate.
            - Include any potential risks or challenges for each recommendation.
            - Reference relevant metrics or insights from the analysis if possible.
            - Use professional, executive-level language.
            - Add an appropriate emoji based on risk level:
                - ✅ for Low risk
                - ⚠️ for Medium risk
                - ❗for High risk

            Output Format:

            ### 📋 Recommendations Table

            | # | Recommendation Title | Expected Impact (%) | Potential Risk (with Emoji) |
            |---|-----------------------|---------------------|-----------------------------|
            | 1 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
            | 2 | [Title] | [Estimated Impact %] | [Emoji] [Main risk] |
            | ... | ... | ... | ... |

            ---

            ### 📋 Full Recommendation Details

            1. **[Recommendation Title]** [Emoji]
            - **Details:** Explain clearly what should be done and why.
            - **Expected Impact:** [e.g., Increase attendance by 10%]
            - **Metrics Reference:** [Reference specific metric if available, e.g., matches with <50% attendance]
            - **Potential Risks:** [Possible challenges or risks involved]
            - **Timeline:** [Short-term or Long-term]

            Repeat similarly for each recommendation.
            '''

            rec_template = PromptTemplate(
                input_variables=["data_info", "data_sample", "data_description", "analysis", "num_recommendations"],
                template=recommendation_prompt
            )

            rec_chain = LLMChain(llm=self.llm, prompt=rec_template)

            rec_response = rec_chain.run(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description,
                analysis=analysis,
                num_recommendations=num_recommendations
            )

            formatted_rec_prompt = recommendation_prompt.format(
                data_info=data_info,
                data_sample=data_sample,
                data_description=data_description,
                analysis=analysis,
                num_recommendations=num_recommendations
            )
            
            self._save_to_memory(formatted_rec_prompt, rec_response)
            return rec_response
            
        except Exception as e:
            error_msg = f"Error generating recommendations: {str(e)}"
            logger.error(error_msg)
            return error_msg 

    def save_analysis(self, filepath=None):
        """
        Save the analysis results to a file.
        
        Args:
            filepath: Path to save the analysis results. If None, generates a timestamped filename.
            
        Returns:
            The path where the analysis was saved
        """
        try:
            if not filepath:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                # Create analysis_results directory if it doesn't exist
                os.makedirs("analysis_results", exist_ok=True)
                filepath = f"analysis_results/analysis_{timestamp}.json"
            
            # Create dictionary of analysis results
            analysis_data = {
                "data_info": self.data_info,
                "data_description": self.data_description,
                "report_id": self.report_id,
                "user_id": self.user_id,
                "timestamp": datetime.now().isoformat()
            }
            
            # Add any generated analyses
            if hasattr(self, 'analysis'):
                analysis_data["analysis"] = self.analysis
                
            # Add any saved memory interactions (excluding actual messages for serialization)
            memory_logs = []
            for i in range(0, len(self.memory), 2):
                if i+1 < len(self.memory):
                    memory_logs.append({
                        "prompt": self.memory[i].content,
                        "response": self.memory[i+1].content
                    })
            analysis_data["memory_logs"] = memory_logs
            
            # Save as JSON file
            with open(filepath, 'w', encoding='utf-8') as f:
                json.dump(analysis_data, f, ensure_ascii=False, indent=2)
                
            logger.info(f"Analysis saved to {filepath}")
            return filepath
            
        except Exception as e:
            logger.error(f"Error saving analysis: {str(e)}")
            return None
            
    def load_analysis(self, filepath):
        """
        Load analysis results from a file.
        
        Args:
            filepath: Path to the saved analysis file
            
        Returns:
            True if loading was successful, False otherwise
        """
        try:
            if not os.path.exists(filepath):
                logger.error(f"Analysis file not found: {filepath}")
                return False
                
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
            # Restore data
            if "data_info" in data:
                self.data_info = data["data_info"]
            if "data_description" in data:
                self.data_description = data["data_description"]
            if "report_id" in data:
                self.report_id = data["report_id"]
            if "user_id" in data:
                self.user_id = data["user_id"]
            if "analysis" in data:
                self.analysis = data["analysis"]
                
            # Restore memory logs
            self.memory = []
            if "memory_logs" in data:
                for log in data["memory_logs"]:
                    self.memory.append(HumanMessage(content=log["prompt"]))
                    self.memory.append(AIMessage(content=log["response"]))
                    
            logger.info(f"Analysis loaded from {filepath}")
            return True
            
        except Exception as e:
            logger.error(f"Error loading analysis: {str(e)}")
            return False
            
    def export_analysis_report(self, output_format="json", filepath=None):
        """
        Export the analysis in various formats.
        
        Args:
            output_format: Format to export (json, txt, html)
            filepath: Path to save the report. If None, generates a timestamped filename.
            
        Returns:
            The path where the report was saved
        """
        try:
            if not hasattr(self, 'analysis'):
                logger.warning("No analysis available to export")
                return None
                
            if not filepath:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                os.makedirs("reports", exist_ok=True)
                filepath = f"reports/analysis_report_{timestamp}.{output_format}"
                
            if output_format == "json":
                with open(filepath, 'w', encoding='utf-8') as f:
                    json.dump({
                        "analysis": self.analysis,
                        "metadata": {
                            "data_info": self.data_info,
                            "report_id": self.report_id,
                            "user_id": self.user_id,
                            "timestamp": datetime.now().isoformat()
                        }
                    }, f, ensure_ascii=False, indent=2)
            
            elif output_format == "txt":
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(f"=== DATA ANALYSIS REPORT ===\n")
                    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
                    f.write(self.analysis)
                    
            elif output_format == "html":
                html_content = f"""
                <!DOCTYPE html>
                <html>
                <head>
                    <title>Data Analysis Report</title>
                    <style>
                        body {{ font-family: Arial, sans-serif; margin: 40px; }}
                        h1 {{ color: #2c3e50; }}
                        .metadata {{ color: #7f8c8d; font-size: 0.9em; }}
                        .analysis {{ margin-top: 20px; line-height: 1.6; }}
                    </style>
                </head>
                <body>
                    <h1>Data Analysis Report</h1>
                    <div class="metadata">
                        <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                        <p>Report ID: {self.report_id or 'N/A'}</p>
                    </div>
                    <div class="analysis">
                        {self.analysis.replace('\n', '<br>')}
                    </div>
                </body>
                </html>
                """
                with open(filepath, 'w', encoding='utf-8') as f:
                    f.write(html_content)
            
            else:
                logger.error(f"Unsupported output format: {output_format}")
                return None
                
            logger.info(f"Analysis report exported to {filepath}")
            return filepath
            
        except Exception as e:
            logger.error(f"Error exporting analysis report: {str(e)}")
            return None 

    def recommend_cleaning_strategy(self) -> Dict:
        """
        Analyze the dataframe and recommend data cleaning strategies.
        
        Returns:
            Dictionary of recommended cleaning strategies
        """
        try:
            df = self.dataframe
            recommendations = {}
            
            # Check for missing values
            missing_counts = df.isna().sum()
            missing_cols = missing_counts[missing_counts > 0]
            
            if len(missing_cols) > 0:
                # Recommend strategy based on column type and missing percentage
                missing_strategy = {}
                for col in missing_cols.index:
                    missing_pct = missing_counts[col] / len(df)
                    
                    if missing_pct > 0.5:
                        # Too many missing values, consider dropping the column
                        missing_strategy[col] = "drop_column"
                    elif df[col].dtype in [np.float64, np.int64]:
                        # For numeric columns, use median
                        missing_strategy[col] = "median"
                    else:
                        # For categorical columns, use mode
                        missing_strategy[col] = "mode"
                
                recommendations['missing_values'] = missing_strategy
            
            # Check for duplicates
            duplicate_count = df.duplicated().sum()
            if duplicate_count > 0:
                recommendations['duplicates'] = 'drop_first'
            
            # Check for outliers in numeric columns
            numeric_cols = df.select_dtypes(include=np.number).columns
            outlier_strategy = {}
            
            for col in numeric_cols:
                # Use IQR to detect outliers
                Q1 = df[col].quantile(0.25)
                Q3 = df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                
                outlier_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                
                if outlier_count > 0 and outlier_count / len(df) < 0.05:
                    # Small percentage of outliers, use IQR clipping
                    outlier_strategy[col] = "iqr"
            
            if outlier_strategy:
                recommendations['outliers'] = outlier_strategy
            
            # Check for special characters in string columns
            string_cols = df.select_dtypes(include=['object']).columns
            special_chars_cols = []
            
            for col in string_cols:
                if df[col].dtype == 'object':
                    # Check for special characters
                    has_special = df[col].astype(str).str.contains(r'[^\w\s]', regex=True).any()
                    if has_special:
                        special_chars_cols.append(col)
            
            if special_chars_cols:
                recommendations['special_chars'] = {
                    'strategy': 'remove',
                    'columns': special_chars_cols
                }
            
            # Check for potential data type conversions
            type_conversions = {}
            
            # Check for datetime columns
            for col in df.columns:
                if df[col].dtype == 'object':
                    # Try to convert to datetime
                    try:
                        pd.to_datetime(df[col], errors='raise')
                        type_conversions[col] = 'datetime'
                    except:
                        pass
            
            if type_conversions:
                recommendations['data_types'] = type_conversions
            
            return recommendations
            
        except Exception as e:
            logger.error(f"Error recommending cleaning strategy: {str(e)}")
            return {}

    def clean_data(self, strategies: Dict = None) -> pd.DataFrame:
        """
        Clean the dataset using various strategies.
        
        Args:
            strategies: Dictionary of cleaning strategies to apply.
                Available strategies:
                - missing_values: 'drop', 'mean', 'median', 'mode', 'zero', 'value'
                - duplicates: 'drop_first', 'drop_last', 'keep'
                - outliers: 'clip', 'remove', 'iqr', 'zscore'
                - special_chars: 'remove', 'replace'
                - data_types: Dict mapping column names to desired data types
                
        Returns:
            Cleaned dataframe
        """
        try:
            # Start with a fresh copy of the original data
            df = self.original_dataframe.copy()
            
            # Default strategies if none provided
            if strategies is None:
                strategies = {
                    'missing_values': 'median',
                    'duplicates': 'drop_first',
                    'outliers': 'iqr',
                    'special_chars': 'remove'
                }
            
            logger.info("Starting data cleaning process")
            self.cleaning_log = []  # Reset cleaning log
            
            # 1. Handle missing values
            if 'missing_values' in strategies:
                strategy = strategies['missing_values']
                missing_count_before = df.isna().sum().sum()
                
                if strategy == 'drop':
                    df = df.dropna()
                    self.cleaning_log.append(f"Dropped {missing_count_before} missing values")
                
                elif strategy in ['mean', 'median', 'mode']:
                    for col in df.select_dtypes(include=np.number).columns:
                        if df[col].isna().sum() > 0:
                            if strategy == 'mean':
                                df[col] = df[col].fillna(df[col].mean())
                            elif strategy == 'median':
                                df[col] = df[col].fillna(df[col].median())
                            elif strategy == 'mode':
                                df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 0)
                    
                    # For non-numeric columns, use mode
                    for col in df.select_dtypes(exclude=np.number).columns:
                        if df[col].isna().sum() > 0:
                            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else "")
                            
                    self.cleaning_log.append(f"Filled {missing_count_before} missing values using {strategy}")
                
                elif strategy == 'zero':
                    df = df.fillna(0)
                    self.cleaning_log.append(f"Filled {missing_count_before} missing values with zero")
                
                elif isinstance(strategy, dict):
                    # Custom value for each column
                    for col, value in strategy.items():
                        if col in df.columns:
                            df[col] = df[col].fillna(value)
                    self.cleaning_log.append(f"Filled missing values with custom values for specified columns")
            
            # 2. Handle duplicates
            if 'duplicates' in strategies:
                strategy = strategies['duplicates']
                duplicate_count = df.duplicated().sum()
                
                if strategy == 'drop_first':
                    df = df.drop_duplicates(keep='first')
                    self.cleaning_log.append(f"Removed {duplicate_count} duplicate rows (keeping first occurrence)")
                
                elif strategy == 'drop_last':
                    df = df.drop_duplicates(keep='last')
                    self.cleaning_log.append(f"Removed {duplicate_count} duplicate rows (keeping last occurrence)")
            
            # 3. Handle outliers
            if 'outliers' in strategies:
                strategy = strategies['outliers']
                numeric_cols = df.select_dtypes(include=np.number).columns
                
                if strategy == 'iqr':
                    # IQR method
                    for col in numeric_cols:
                        Q1 = df[col].quantile(0.25)
                        Q3 = df[col].quantile(0.75)
                        IQR = Q3 - Q1
                        lower_bound = Q1 - 1.5 * IQR
                        upper_bound = Q3 + 1.5 * IQR
                        
                        outliers_count = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
                        if outliers_count > 0:
                            df.loc[df[col] < lower_bound, col] = lower_bound
                            df.loc[df[col] > upper_bound, col] = upper_bound
                            self.cleaning_log.append(f"Clipped {outliers_count} outliers in column '{col}' using IQR method")
                
                elif strategy == 'zscore':
                    # Z-score method
                    from scipy import stats
                    for col in numeric_cols:
                        z_scores = np.abs(stats.zscore(df[col].fillna(df[col].median())))
                        outliers = z_scores > 3
                        outliers_count = outliers.sum()
                        
                        if outliers_count > 0:
                            df.loc[outliers, col] = df[col].median()
                            self.cleaning_log.append(f"Replaced {outliers_count} outliers in column '{col}' using Z-score method")
                
                elif strategy == 'remove':
                    for col in numeric_cols:
                        Q1 = df[col].quantile(0.25)
                        Q3 = df[col].quantile(0.75)
                        IQR = Q3 - Q1
                        lower_bound = Q1 - 1.5 * IQR
                        upper_bound = Q3 + 1.5 * IQR
                        
                        outlier_mask = (df[col] < lower_bound) | (df[col] > upper_bound)
                        outliers_count = outlier_mask.sum()
                        
                        if outliers_count > 0:
                            df = df[~outlier_mask]
                            self.cleaning_log.append(f"Removed {outliers_count} rows with outliers in column '{col}'")
            
            # 4. Handle special characters
            if 'special_chars' in strategies:
                strategy = strategies['special_chars']
                string_cols = df.select_dtypes(include=['object']).columns
                
                if strategy == 'remove':
                    for col in string_cols:
                        if df[col].dtype == 'object':
                            # Replace special characters with empty string
                            df[col] = df[col].astype(str).str.replace(r'[^\w\s]', '', regex=True)
                    self.cleaning_log.append(f"Removed special characters from text columns")
                
                elif strategy == 'replace':
                    for col in string_cols:
                        if df[col].dtype == 'object':
                            # Replace special characters with underscore
                            df[col] = df[col].astype(str).str.replace(r'[^\w\s]', '_', regex=True)
                    self.cleaning_log.append(f"Replaced special characters with underscore in text columns")
            
            # 5. Handle data types
            if 'data_types' in strategies and isinstance(strategies['data_types'], dict):
                type_conversions = strategies['data_types']
                
                for col, dtype in type_conversions.items():
                    if col in df.columns:
                        try:
                            if dtype == 'datetime':
                                df[col] = pd.to_datetime(df[col], errors='coerce')
                            else:
                                df[col] = df[col].astype(dtype)
                            self.cleaning_log.append(f"Converted column '{col}' to {dtype} type")
                        except Exception as e:
                            self.cleaning_log.append(f"Failed to convert column '{col}' to {dtype}: {str(e)}")
            
            # Update the dataframe and derived properties
            rows_diff = len(self.dataframe) - len(df)
            cols_diff = len(self.dataframe.columns) - len(df.columns)
            
            self.dataframe = df
            
            # Update derived properties
            self.data_info = data_infer(df)
            self.data_description = data_describer(df)
            self.data_sample = df.head().to_string()
            self.data_cols = ", ".join(df.columns)
            
            logger.info(f"Data cleaning completed. Rows changed: {rows_diff}, Columns changed: {cols_diff}")
            summary = f"Data cleaning completed. Original shape: {self.original_dataframe.shape}, New shape: {df.shape}"
            self.cleaning_log.append(summary)
            
            return df
            
        except Exception as e:
            error_msg = f"Error cleaning data: {str(e)}"
            logger.error(error_msg)
            self.cleaning_log.append(error_msg)
            return self.dataframe

    def restore_original_data(self) -> pd.DataFrame:
        """
        Restore the dataframe to its original state before cleaning.
        
        Returns:
            Original dataframe
        """
        try:
            self.dataframe = self.original_dataframe.copy()
            
            # Update derived properties
            self.data_info = data_infer(self.dataframe)
            self.data_description = data_describer(self.dataframe)
            self.data_sample = self.dataframe.head().to_string()
            self.data_cols = ", ".join(self.dataframe.columns)
            
            logger.info("Restored original dataframe")
            return self.dataframe
            
        except Exception as e:
            logger.error(f"Error restoring original data: {str(e)}")
            return None

    def get_cleaning_log(self) -> List[str]:
        """
        Get the log of data cleaning operations performed.
        
        Returns:
            List of cleaning operations
        """
        return self.cleaning_log 